# PRAGYAAN – Unit 5
Machine Learning project (21CSC305P). Data: English subset of the Kaggle Multilingual Customer Support Tickets dataset.

In [ ]:
# Upload tickets.parquet and emb.npy (from the zip) to the Colab session first, then run all cells.
!pip -q install xgboost hmmlearn pyarrow
import os; os.makedirs('/content/report_work/fig', exist_ok=True)
!mv -f /content/tickets.parquet /content/emb.npy /content/report_work/ 2>/dev/null; ls /content/report_work

In [ ]:
%%writefile /content/common.py
"""Shared data loading, split and plotting style for the PRAGYAAN report experiments."""
import re
import sys
import time

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

import re as _re
_PH = _re.compile(r"<(?:tel_num|email|url|account_num|order_num|ip|date|name|address|password|username|phone|amount|id)[^>]*>", _re.I)


def clean_text(text, strict=False):
    out = _re.sub(r"\s+", " ", _PH.sub(" ", text or "")).strip()
    return out.lower() if strict else out

W = "/content/report_work"
FIG = f"{W}/fig"
LABELS = ["high", "medium", "low"]
SEED = 42

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.bbox": "tight",
})
C = {"high": "#B23A48", "medium": "#E0A458", "low": "#4E8098", "ink": "#2B2D42", "grey": "#8D99AE"}


def load():
    df = pd.read_parquet(f"{W}/tickets.parquet")
    for c in ["text", "priority", "type", "queue", "answer"]:
        df[c] = np.array(df[c].tolist(), dtype=object)
    emb = np.load(f"{W}/emb.npy")
    df["clean"] = [clean_text(t, strict=True) for t in df["text"]]
    df["words"] = df["text"].str.split().str.len()
    return df, emb


def split(df):
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=0.2, stratify=df["priority"], random_state=SEED)
    return tr, te


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words="english", min_df=2)


def evaluate(name, model, Xtr, ytr, Xte, yte, proba=True):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_s = time.perf_counter() - t0
    pred = model.predict(Xte)
    p, r, f, s = precision_recall_fscore_support(yte, pred, labels=LABELS, zero_division=0)
    out = {"model": name, "accuracy": accuracy_score(yte, pred), "macro_f1": f1_score(yte, pred, average="macro"),
           "weighted_f1": f1_score(yte, pred, average="weighted"), "fit_seconds": fit_s,
           "per_class": {lab: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                         for i, lab in enumerate(LABELS)}}
    if proba and hasattr(model, "predict_proba"):
        try:
            P = model.predict_proba(Xte)
            out["roc_auc_macro"] = roc_auc_score(yte, P, multi_class="ovr", average="macro", labels=list(model.classes_))
        except Exception:
            pass
    elif hasattr(model, "decision_function"):
        D = model.decision_function(Xte)
        out["roc_auc_macro"] = _auc_from_scores(yte, D, list(model.classes_))
    out["pred"] = list(pred)
    return out


def _auc_from_scores(y, D, classes):
    from scipy.special import softmax
    return roc_auc_score(y, softmax(D, axis=1), multi_class="ovr", average="macro", labels=classes)


def words_re(s):
    return re.findall(r"[a-z]+", s.lower())


In [ ]:
"""Unit 5: CART, Random Forest (bagging), AdaBoost, gradient boosting (XGBoost) and a soft-voting ensemble."""
import json
import time

import numpy as np
import sys; sys.path.insert(0, '/content')
from common import FIG, LABELS, SEED, C, W, evaluate, load, plt, split, tfidf
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

df, emb = load()
tr, te = split(df)
vec = tfidf()
Xtr = vec.fit_transform(df["clean"].iloc[tr]); Xte = vec.transform(df["clean"].iloc[te])
ytr = df["priority"].iloc[tr].to_numpy(dtype=str); yte = df["priority"].iloc[te].to_numpy(dtype=str)
enc = {lab: i for i, lab in enumerate(LABELS)}; dec = np.array(LABELS)
R = {}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

# ---------------- CART: tune depth and leaf size (5-fold CV)
cart_grid = {}
for d in (10, 20, 40, None):
    for leaf in (1, 5, 20):
        m = DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, class_weight="balanced", random_state=SEED)
        cart_grid[f"{d}|{leaf}"] = cross_val_score(m, Xtr, ytr, cv=cv, scoring="f1_macro").mean()
best_cart = max(cart_grid, key=cart_grid.get)
R["cart_grid"] = cart_grid; R["cart_best"] = best_cart
d, leaf = best_cart.split("|"); d = None if d == "None" else int(d); leaf = int(leaf)

# ---------------- Random forest: number of trees and features per split (5-fold CV)
rf_grid = {}
for n in (100, 200, 400):
    for mf in ("sqrt", 0.1):
        m = RandomForestClassifier(n_estimators=n, max_features=mf, class_weight="balanced", n_jobs=-1, random_state=SEED)
        rf_grid[f"{n}|{mf}"] = cross_val_score(m, Xtr, ytr, cv=cv, scoring="f1_macro").mean()
best_rf = max(rf_grid, key=rf_grid.get)
R["rf_grid"] = rf_grid; R["rf_best"] = best_rf
n_best, mf_best = best_rf.split("|"); n_best = int(n_best); mf_best = mf_best if mf_best == "sqrt" else float(mf_best)

# ---------------- AdaBoost (SAMME) on shallow trees
ada_grid = {}
for depth in (1, 3):
    m = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=depth), n_estimators=200, learning_rate=0.5,
                           random_state=SEED)
    ada_grid[f"{depth}|0.5"] = cross_val_score(m, Xtr, ytr, cv=StratifiedKFold(3, shuffle=True, random_state=SEED),
                                               scoring="f1_macro").mean()
    print("Ada", depth, round(ada_grid[f"{depth}|0.5"], 4), flush=True)
best_ada = max(ada_grid, key=ada_grid.get); R["ada_grid"] = ada_grid; R["ada_best"] = best_ada
a_depth, a_lr = best_ada.split("|"); a_depth = int(a_depth); a_lr = float(a_lr)

# ---------------- XGBoost (gradient boosting) with balanced sample weights
ytr_i = np.array([enc[y] for y in ytr])
sw = compute_sample_weight("balanced", ytr)
xgb_grid = {}
for depth in (4, 8):
    for eta in (0.1,):
        scores = []
        for a, b in StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xtr, ytr):
            m = XGBClassifier(n_estimators=400, max_depth=depth, learning_rate=eta, subsample=0.8, colsample_bytree=0.5,
                              tree_method="hist", objective="multi:softprob", eval_metric="mlogloss", n_jobs=2,
                              random_state=SEED)
            m.fit(Xtr[a], ytr_i[a], sample_weight=sw[a])
            scores.append(f1_score(ytr_i[b], m.predict(Xtr[b]), average="macro"))
        xgb_grid[f"{depth}|{eta}"] = float(np.mean(scores))
        print("XGB", depth, eta, round(xgb_grid[f"{depth}|{eta}"], 4), flush=True)
best_xgb = max(xgb_grid, key=xgb_grid.get); R["xgb_grid"] = xgb_grid; R["xgb_best"] = best_xgb
x_depth, x_eta = best_xgb.split("|"); x_depth = int(x_depth); x_eta = float(x_eta)


class XGBWrap(XGBClassifier):
    """XGBoost with string labels and balanced weights, so it fits the same evaluation code."""

    def fit(self, X, y, **kw):
        yi = np.array([enc[v] for v in y])
        super().fit(X, yi, sample_weight=compute_sample_weight("balanced", y))
        self.classes_str_ = dec
        return self

    def predict(self, X):
        return dec[super().predict(X)]


def xgb():
    return XGBWrap(n_estimators=400, max_depth=x_depth, learning_rate=x_eta, subsample=0.8, colsample_bytree=0.5,
                   tree_method="hist", objective="multi:softprob", eval_metric="mlogloss", n_jobs=2, random_state=SEED)


models = {
    "CART (decision tree)": DecisionTreeClassifier(max_depth=d, min_samples_leaf=leaf, class_weight="balanced", random_state=SEED),
    "Random Forest (bagging)": RandomForestClassifier(n_estimators=n_best, max_features=mf_best, class_weight="balanced",
                                                      n_jobs=-1, random_state=SEED, oob_score=True),
    "AdaBoost": AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=a_depth),
                                   n_estimators=200, learning_rate=a_lr, random_state=SEED),
    "XGBoost (gradient boosting)": xgb(),
}
results = []
for name, m in models.items():
    r = evaluate(name, m, Xtr, ytr, Xte, yte, proba=name != "XGBoost (gradient boosting)")
    if name == "XGBoost (gradient boosting)":
        from sklearn.metrics import roc_auc_score
        P = XGBClassifier.predict_proba(m, Xte)
        r["roc_auc_macro"] = roc_auc_score([enc[y] for y in yte], P, multi_class="ovr", average="macro")
    results.append(r)
    print(f"{name:28s} acc={r['accuracy']:.4f} f1={r['macro_f1']:.4f} auc={r.get('roc_auc_macro', 0):.4f} t={r['fit_seconds']:.1f}s")
rf = models["Random Forest (bagging)"]; R["rf_oob"] = float(rf.oob_score_)

# soft-voting ensemble (probability averaging) of LR, RF and XGBoost


class XGBProba(XGBWrap):
    @property
    def classes_(self):
        return dec

    @classes_.setter
    def classes_(self, v):
        pass

    def predict_proba(self, X):
        return XGBClassifier.predict_proba(self, X)


t0 = time.perf_counter()
lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(Xtr, ytr)
xg = models["XGBoost (gradient boosting)"]
P = (lr.predict_proba(Xtr[:1]) is not None)
order = list(lr.classes_)  # alphabetical: high, low, medium
P_lr = lr.predict_proba(Xte)
P_rf = rf.predict_proba(Xte)[:, [list(rf.classes_).index(c) for c in order]]
P_xg = XGBClassifier.predict_proba(xg, Xte)[:, [LABELS.index(c) for c in order]]
P_avg = (P_lr + P_rf + P_xg) / 3
pred = np.array(order)[P_avg.argmax(1)]
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score  # noqa: E402
p, rr, f, s = precision_recall_fscore_support(yte, pred, labels=LABELS, zero_division=0)
results.append({"model": "Soft-voting ensemble (LR + RF + XGBoost)", "accuracy": accuracy_score(yte, pred),
                "macro_f1": f1_score(yte, pred, average="macro"), "weighted_f1": f1_score(yte, pred, average="weighted"),
                "fit_seconds": time.perf_counter() - t0, "roc_auc_macro": roc_auc_score(yte, P_avg, multi_class="ovr", labels=order),
                "per_class": {lab: {"precision": p[i], "recall": rr[i], "f1": f[i], "support": int(s[i])} for i, lab in enumerate(LABELS)},
                "pred": list(pred)})
print("ensemble", round(results[-1]["accuracy"], 4), round(results[-1]["macro_f1"], 4))

# feature importance of the forest
terms = np.array(vec.get_feature_names_out())
imp = rf.feature_importances_; top = np.argsort(-imp)[:20]
R["rf_top_features"] = [(terms[i], float(imp[i])) for i in top]
fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.barh(terms[top][::-1], imp[top][::-1], color=C["ink"])
ax.set_xlabel("Mean decrease in impurity (Random Forest)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig4_3_rf_importance.png"); plt.close()

# number of trees vs out-of-bag error
oob = []
for n in (25, 50, 100, 200, 400):
    m = RandomForestClassifier(n_estimators=n, max_features=mf_best, class_weight="balanced", n_jobs=-1,
                               random_state=SEED, oob_score=True).fit(Xtr, ytr)
    oob.append((n, 1 - m.oob_score_))
R["oob_curve"] = oob
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot([a for a, _ in oob], [b * 100 for _, b in oob], "o-", color=C["ink"])
ax.set_xlabel("Number of trees"); ax.set_ylabel("Out-of-bag error (%)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig4_4_oob.png"); plt.close()

best = max(results, key=lambda x: x["macro_f1"])
cm = confusion_matrix(yte, best["pred"], labels=LABELS); R["best_unit5"] = best["model"]; R["best_unit5_cm"] = cm.tolist()
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap="Greys")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() * 0.55 else "black")
ax.set_xticks(range(3)); ax.set_xticklabels(LABELS); ax.set_yticks(range(3)); ax.set_yticklabels(LABELS)
ax.set_xlabel("Predicted priority"); ax.set_ylabel("True priority"); ax.spines[:].set_visible(False)
plt.tight_layout(); plt.savefig(f"{FIG}/fig4_5_confusion_best.png"); plt.close()

for x in results:
    x.pop("pred", None)
R["results"] = results
json.dump(R, open(f"{W}/unit5.json", "w"), indent=1, default=float)
print("best", best["model"])
